In [2]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
# !mkdir ../data/Gutenberg
# !unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
# !rm ../data/Gutenberg.zip

--2025-12-14 23:24:00--  https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 54.228.176.245, 54.228.56.77, 52.19.64.37, ...
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|54.228.176.245|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2025-12-14 23:24:01 ERROR 403: Forbidden.

mkdir: cannot create directory ‘../data/Gutenberg’: File exists
Archive:  ../data/Gutenberg.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ../data/Gutenberg.zip or
        ../data/Gutenberg.zip.zip, and cannot find ../data/Gutenberg.zip.ZIP, period.


In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [4]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [5]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.1},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.002,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 1000,
    "weight_decay": 5e-3,
    "diff_threshold": 0.05,
    "adam_betas": (0.9, 0.999)
}


In [6]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)

In [7]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="gutenberg",
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gcof/gutenberg/d2263607b8944c98a3e164d298576f9b



In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'], betas=hyperparams['adam_betas'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion,optimizer, hyperparams, device, experiment)

100%|██████████| 176/176 [00:00<00:00, 345.64it/s]


Epoch 1/256, Train Loss: 1.7680, Val Loss: 1.7481, Val Accuracy: 0.2727


100%|██████████| 176/176 [00:00<00:00, 335.15it/s]


Epoch 2/256, Train Loss: 1.7310, Val Loss: 1.7284, Val Accuracy: 0.2844


100%|██████████| 176/176 [00:00<00:00, 343.19it/s]


Epoch 3/256, Train Loss: 1.7146, Val Loss: 1.7188, Val Accuracy: 0.2864


... (504 outputs left out)

100%|██████████| 176/176 [01:46<00:00,  1.65it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (5674915 > 1048576). Falling back to PNG format.


Epoch 255/256, Train Loss: 1.2296, Val Loss: 1.7303, Val Accuracy: 0.3467


100%|██████████| 176/176 [01:46<00:00,  1.65it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (5676275 > 1048576). Falling back to PNG format.


Epoch 256/256, Train Loss: 1.2304, Val Loss: 1.7315, Val Accuracy: 0.3471


In [15]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : trash
COMET INFO:     url                   : https://www.comet.com/gcof/gutenberg/d2263607b8944c98a3e164d298576f9b
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     acc amount [256]                     : (1.705264610834494e-07, 9.965706447187929e-05)
COMET INFO:     expanding_adam_exp_avg_sq_max [256]  : (5.702006546925986e-06, 0.00048370586591772735)
COMET INFO:     expanding_adam_exp_avg_sq_mean [256] : (2.0710109538413235e-07, 1.9328474081703462e-05)
COMET INFO:     lr [256]                             : (0.00025, 0.001)
COMET INFO:     n_params over train_time [256]       : (4858.877079125556, 395863.5777062131)
COMET INFO:     params am

In [12]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [13]:
accuracy, params

(0.36466666666666664, 1989916)

In [14]:
torch.save(sparse_model, 'gutenberg.pt')